# Lab 6: Interactive Visual Analytics with Folium

In [1]:
import os, json, warnings
warnings.filterwarnings('ignore')
os.makedirs('figs', exist_ok=True)
os.makedirs('results', exist_ok=True)

def save_result(name, value):
    path = 'results/results.json'
    data = json.load(open(path)) if os.path.exists(path) else {}
    data[name] = value
    json.dump(data, open(path, 'w'), indent=2, default=str)

import folium
import pandas as pd
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon
from math import sin, cos, sqrt, atan2, radians
spacex_df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv')
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


### Task 1: Mark all launch sites on a map

In [2]:
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=4)
for _, site in launch_sites_df.iterrows():
    coord = [site['Lat'], site['Long']]
    folium.Circle(coord, radius=1000, color='#d35400', fill=True).add_child(folium.Popup(site['Launch Site'])).add_to(site_map)
    folium.map.Marker(coord, icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
        html=f"<div style='font-size:13px;color:#d35400;white-space:nowrap;'><b>{site['Launch Site']}</b></div>")).add_to(site_map)
site_map.save('figs/map_sites.html')
site_map

### Task 2: Mark successful (green) and failed (red) launches for each site

In [3]:
spacex_df['marker_color'] = spacex_df['class'].map({1: 'green', 0: 'red'})
site_map = folium.Map(location=[28.5623, -80.5774], zoom_start=5)
marker_cluster = MarkerCluster().add_to(site_map)
for _, record in spacex_df.iterrows():
    folium.Marker(location=[record['Lat'], record['Long']],
                  icon=folium.Icon(color='white', icon_color=record['marker_color']),
                  popup=f"{record['Launch Site']} - {'Success' if record['class'] else 'Failure'}").add_to(marker_cluster)
site_counts = spacex_df.groupby('Launch Site')['class'].agg(['sum', 'count'])
save_result('folium_site_success', {k: f"{int(v['sum'])}/{int(v['count'])}" for k, v in site_counts.iterrows()})
site_map.save('figs/map_clusters.html')
site_map

### Task 3: Distances from a launch site to its proximities
Coordinates of nearby points were read from the map with the MousePosition tool; road positions were confirmed with OpenStreetMap reverse geocoding.

In [4]:
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon, dlat = lon2 - lon1, lat2 - lat1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

site = [28.56319718, -80.57682003]            # CCAFS SLC-40
points = {
    'Coastline': [28.56320, -80.56785],
    'Highway (Samuel C Phillips Pkwy)': [28.56321, -80.57076],
    'Access road (Titan III Rd)': [28.56336, -80.58697],
    'City (Titusville)': [28.61208, -80.80764],
}
site_map = folium.Map(location=site, zoom_start=15)
site_map.add_child(MousePosition(position='topright', separator=' Long: ', prefix='Lat:', num_digits=5))
folium.Marker(site, popup='CCAFS SLC-40').add_to(site_map)
distances = {}
label_offset = {'Coastline': (-10, 30), 'Highway (Samuel C Phillips Pkwy)': (-10, -8)}
for name, coord in points.items():
    d = calculate_distance(site[0], site[1], coord[0], coord[1])
    distances[name] = round(d, 2)
    folium.Marker(coord, icon=DivIcon(icon_size=(20, 20), icon_anchor=label_offset.get(name, (0, 0)),
        html=f"<div style='font-size:13px;color:#d35400;white-space:nowrap;background:rgba(255,255,255,.8);padding:1px 4px;'><b>{name}: {d:.2f} km</b></div>")).add_to(site_map)
    folium.PolyLine([site, coord], weight=2, color='#2b6cb0').add_to(site_map)
save_result('proximity_km', distances)
site_map.save('figs/map_proximity.html')
print(distances)
site_map

{'Coastline': 0.88, 'Highway (Samuel C Phillips Pkwy)': 0.59, 'Access road (Titan III Rd)': 0.99, 'City (Titusville)': 23.19}
